# Constraints: Lagrange Multipliers and Baumgarte Stabilization

**MechanicsDSL Notebook 03** | Classical Mechanics | Intermediate

This notebook demonstrates holonomic constraint handling in MechanicsDSL: a bead constrained to a circular wire and a pendulum with a rigid rod constraint. We show how MechanicsDSL applies Lagrange multipliers symbolically and prevents constraint drift via Baumgarte stabilization.

**Key concepts:**
- Augmented Lagrangian: $L' = L + \sum_i \lambda_i g_i$
- Constraint forces from Euler-Lagrange applied to $L'$
- Baumgarte stabilization: $\ddot{g} + 2\alpha\dot{g} + \beta^2 g = 0$

**MechanicsDSL DSL specification:**
```
\system{bead_on_wire}
\parameter{m}{1.0}{kg}
\parameter{R}{1.0}{m}
\lagrangian{0.5*m*(\dot{x}^2 + \dot{y}^2) - m*g*y}
\constraint{x^2 + y^2 - R^2}
\target{python_numpy}
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## Theory: Bead on a Circular Wire

For a bead of mass $m$ constrained to a circle of radius $R$ in a gravitational field, the constraint is $g = x^2 + y^2 - R^2 = 0$.

In polar coordinates ($x = R\sin\theta$, $y = -R\cos\theta$), the constraint is automatically satisfied and the system reduces to a simple pendulum. We use this to validate the Cartesian constrained simulation.

In [ ]:
m, R, g = 1.0, 1.0, 9.81
alpha_b, beta_b = 7.0, 7.0  # Baumgarte parameters

def bead_constrained(t, state):
    """Bead on circular wire with Baumgarte stabilization.
    MechanicsDSL-generated from: \lagrangian + \constraint{x^2+y^2-R^2}
    State: [x, y, vx, vy]
    """
    x, y, vx, vy = state
    # Constraint and its derivatives
    g_val  = x**2 + y**2 - R**2
    g_dot  = 2*x*vx + 2*y*vy
    # Lagrange multiplier from solving constrained system
    lam = (-m*g - m*(vx**2+vy**2)/R**2 - 2*alpha_b*m*g_dot - beta_b**2*m*g_val) / (2*m*(x**2+y**2)/m)
    # Accelerations with constraint force + Baumgarte
    ax = (2*x*lam)/m
    ay = -g + (2*y*lam)/m
    return [vx, vy, ax, ay]

# Initial conditions: bead at theta=0.3 rad
theta0 = 0.3
state0 = [R*np.sin(theta0), -R*np.cos(theta0), 0.0, 0.0]
t_eval = np.arange(0, 20, 0.005)
sol = solve_ivp(bead_constrained, [0, 20], state0, t_eval=t_eval, method='DOP853', rtol=1e-10, atol=1e-12)

# Reference: simple pendulum
def pendulum_ref(t, y): return [y[1], -(g/R)*np.sin(y[0])]
sol_ref = solve_ivp(pendulum_ref, [0,20], [theta0, 0.0], t_eval=t_eval, method='DOP853', rtol=1e-10, atol=1e-12)

# Constraint violation
constraint_violation = np.sqrt(sol.y[0]**2 + sol.y[1]**2) - R
theta_cart = np.arctan2(sol.y[0], -sol.y[1])

fig, axes = plt.subplots(3, 1, figsize=(11,9))
fig.suptitle('Bead on Circular Wire — Constrained Cartesian vs Pendulum Reference', fontweight='bold')
axes[0].plot(sol.t, theta_cart, label='Constrained (Cartesian)')
axes[0].plot(sol_ref.t, sol_ref.y[0], '--', alpha=0.7, label='Reference (pendulum)')
axes[0].set_ylabel('Angle (rad)'); axes[0].legend()
axes[1].plot(sol.y[0], sol.y[1], lw=0.5)
theta_circ = np.linspace(0, 2*np.pi, 300)
axes[1].plot(R*np.cos(theta_circ), R*np.sin(theta_circ), 'r--', alpha=0.4, label='Constraint circle')
axes[1].set_aspect('equal'); axes[1].set_xlabel('x (m)'); axes[1].set_ylabel('y (m)'); axes[1].legend()
axes[2].semilogy(sol.t, np.abs(constraint_violation), color='tab:red')
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel(r'$|\sqrt{x^2+y^2} - R|$ (m)')
axes[2].set_title(f'Constraint violation (Baumgarte: α=β={alpha_b})')
plt.tight_layout(); plt.show()
print(f'Max constraint violation: {np.max(np.abs(constraint_violation)):.2e} m')

## Baumgarte Parameter Study

The parameters $\alpha, \beta$ control how aggressively constraint violations are driven to zero. Too small → drift accumulates. Too large → numerical stiffness.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for ab in [1.0, 5.0, 10.0, 20.0]:
    def bead_ab(t, state, ab=ab):
        x, y, vx, vy = state
        g_val = x**2+y**2-R**2; g_dot = 2*x*vx+2*y*vy
        lam = (-m*g - m*(vx**2+vy**2)/R**2 - 2*ab*m*g_dot - ab**2*m*g_val)/(2*m*(x**2+y**2)/m)
        return [vx, vy, 2*x*lam/m, -g+2*y*lam/m]
    s = solve_ivp(bead_ab, [0,20], state0, t_eval=t_eval, method='RK45', rtol=1e-8, atol=1e-10)
    viol = np.abs(np.sqrt(s.y[0]**2+s.y[1]**2)-R)
    ax.semilogy(s.t, viol, lw=0.8, label=fr'$\alpha=\beta={ab}$')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Constraint violation (m)')
ax.set_title('Baumgarte parameter study'); ax.legend()
plt.tight_layout(); plt.show()

## Summary
| Method | Constraint drift | Numerical stiffness |
|--------|-----------------|--------------------|
| No stabilization | Unbounded growth | Low |
| Baumgarte (α=β=5) | Exponential decay | Moderate |
| Baumgarte (α=β=20) | Faster decay | Higher |

MechanicsDSL applies Baumgarte stabilization symbolically before code generation — all backends receive stable constraint forces automatically.

**Next:** [04 — Rigid Body Dynamics: Euler Angles and the Symmetric Top](04_rigid_body.ipynb)